# 从 NumPy 到 CUDA：CuPy 数据边界与计时实验

本文配套 Notebook 整理自本机 CuPy 学习实验，源仓库基线为 `42f88530a54d3db846bee6bcf8d656554e90ec75`。

需要本机或远程的 Python CUDA Kernel。博客中的 JupyterLite/Pyodide 无法运行这些原生 CUDA 单元。
按顺序执行全部单元；没有设备时 SKIPPED 不代表 GPU 验证通过。本文保存的下载版不含机器路径、执行输出或性能承诺。


## 1. 环境：分开检查 Python 包与 CUDA 设备

NumPy 与 pandas 在 CPU 上运行。CuPy 的导入、设备发现和真实运算是三个不同的检查点。没有 CuPy 或没有可访问的 GPU 时，CPU 单元仍可运行；GPU 单元会明确输出 SKIPPED。


In [ ]:
import sys  # 记录执行 Notebook 的 Python 版本。
import time  # 为含数据传输的端到端路径使用墙钟计时。
import numpy as np  # 作为 CPU 数组与数值正确性的参照实现。
import pandas as pd  # 构造一个小型表格计算示例。

cp = None  # 未安装 CuPy 时保留显式状态，避免后续误用。
gpu_ready = False  # 设备可见不等于后续算子一定执行成功。
gpu_verified = False  # 只有真实 GPU 运算与 NumPy 比较通过后才设为 True。
gpu_reason = ""  # 保存导入或设备发现失败的原因。

try:  # 只在环境检查阶段处理缺少依赖的情况。
    import cupy as cp  # 加载 CuPy；此操作本身不证明 CUDA 可用。
except ImportError as exc:  # 没有 CuPy 时仍允许 CPU 基线运行。
    gpu_reason = f"CuPy 导入失败：{exc}"
else:  # CuPy 已导入，继续检查当前进程可见的设备。
    try:
        device_count = cp.cuda.runtime.getDeviceCount()  # 返回可访问的 CUDA 设备数。
        gpu_ready = device_count > 0  # 至少有一个设备才能进入 GPU 单元。
        if not gpu_ready:
            gpu_reason = "当前进程没有可见的 CUDA 设备。"
    except cp.cuda.runtime.CUDARuntimeError as exc:  # 记录 CUDA 初始化错误。
        gpu_reason = f"CUDA 设备检查失败：{exc}"

print("Python:", sys.version.split()[0])  # 输出版本，避免保存本机绝对路径。
print("NumPy:", np.__version__, "| pandas:", pd.__version__)  # 输出 CPU 依赖版本。
print("CuPy:", cp.__version__ if cp is not None else "未安装")  # 输出实际加载的 CuPy。
print("GPU 可见:", gpu_ready, "| 原因:", gpu_reason or "无")  # 同时显示状态与原因。

## 2. CPU 基线：shape、dtype 与期望结果

用 3×4 的 float32 矩阵建立一个可手算的基线。pandas 的列计算仍然在 CPU，后续只把显式选出的数值列送到 GPU。


In [ ]:
a_cpu = np.arange(12, dtype=np.float32).reshape(3, 4)  # 创建 0..11，并改为 3 行 4 列。
expected = a_cpu @ a_cpu.T  # (3,4) @ (4,3) 得到 (3,3) 的 CPU 参照结果。
sales = pd.DataFrame({"quantity": [2, 5, 3], "price": [10.0, 8.0, 15.0]})  # 创建 CPU 表格。
sales["revenue"] = sales["quantity"] * sales["price"]  # 逐行计算收入，不会调用 CUDA。
revenue_cpu = sales["revenue"].to_numpy(dtype=np.float32)  # 取出数值列并明确为 float32。
print("CPU 矩阵乘法：\n", expected)  # 打印可以人工核对的矩阵。
print("收入合计：", revenue_cpu.sum())  # CPU 求和应为 105。

## 3. 设备边界：上传、计算、回传与数值校验

本单元使用 blocking=True 简化主机数组的生命周期。后续运算若报错会让单元失败，不会把 GPU 执行失败包装成 SKIPPED。


In [ ]:
if gpu_ready:  # 只有设备发现成功才尝试真实 GPU 运算。
    a_gpu = cp.asarray(a_cpu, blocking=True)  # 把 CPU 数组复制到当前 GPU，并等待上传完成。
    product_gpu = a_gpu @ a_gpu.T  # 在 GPU 上计算 3×3 矩阵乘积。
    result_cpu = cp.asnumpy(product_gpu)  # 默认 blocking=True，回传并等待 CPU 结果可用。
    np.testing.assert_allclose(result_cpu, expected, rtol=1e-5, atol=1e-6)  # 对比绝对与相对误差。
    revenue_gpu = cp.asarray(revenue_cpu, blocking=True)  # 只上传选出的数值列。
    total_gpu = revenue_gpu.sum()  # 在 GPU 上归约得到标量数组。
    total_cpu = cp.asnumpy(total_gpu)  # 读取标量也需要同步与设备到主机的传输。
    np.testing.assert_allclose(total_cpu, revenue_cpu.sum(), rtol=1e-5, atol=1e-6)  # 校验求和。
    gpu_verified = True  # 只在两项真实 GPU 校验完成后记录通过。
    print("PASS: GPU 矩阵乘法与归约结果均匹配 NumPy。")
else:
    print("SKIPPED:", gpu_reason)  # 跳过只说明设备不可用，不是 GPU 测试通过。

## 4. RawKernel：从线程坐标映射到数组元素

RawKernel 默认使用 NVRTC 在第一次调用时编译。这里的 CUDA 函数只接受连续的 float32 一维数组；原始指针不携带 shape、dtype 或 strides。256 个线程是一组实验参数，不是通用最优值。


In [ ]:
if gpu_verified:  # 前一单元必须先证明真实运算可用。
    vector_add = cp.RawKernel(
        r"""
        extern "C" __global__  // 固定导出名称，并声明为 GPU kernel。
        void vector_add(const float* x, const float* y, float* out, int n) {
            int i = blockIdx.x * blockDim.x + threadIdx.x;  // 计算线程的一维全局编号。
            if (i < n) {  // 最后一组线程可能超过数组长度，必须检查边界。
                out[i] = x[i] + y[i];  // 每个有效线程只写自己负责的元素。
            }
        }
        """,
        "vector_add",  # 函数名必须与 CUDA 源码中的导出名称相同。
    )
    n = 10003  # 刻意选择不能被 256 整除的长度。
    threads = 256  # 每个 Block 的线程数。
    blocks = (n + threads - 1) // threads  # 向上取整，保证最后几个元素也被覆盖。
    x_gpu = cp.arange(n, dtype=cp.float32)  # 直接在 GPU 上创建连续的一维输入。
    y_gpu = cp.ones(n, dtype=cp.float32)  # 创建相同 dtype 的第二个输入。
    out_gpu = cp.empty_like(x_gpu)  # 分配输出但不清零，kernel 必须写完所有有效位置。
    vector_add((blocks,), (threads,), (x_gpu, y_gpu, out_gpu, np.int32(n)))  # grid、block、参数元组。
    expected_add = np.arange(n, dtype=np.float32) + 1  # CPU 端建立参照结果。
    np.testing.assert_allclose(cp.asnumpy(out_gpu), expected_add)  # 回传同时等待 kernel 完成。
    print("PASS: RawKernel 向量加法，40 个 Block、10240 个线程、10003 个有效元素。")
else:
    print("SKIPPED: RawKernel 需要先通过真实 GPU 校验。")

## 5. 边界复查：空数组与非整 Block 长度

空数组不启动零大小 Grid；其他长度用同一个已编译 kernel 验证。该检查覆盖长度边界，不证明任意 dtype 或非连续视图也适用。


In [ ]:
if gpu_verified:  # 重用上一个单元创建的 vector_add。
    for test_n in (0, 1, 255, 256, 257, 10003):  # 覆盖空数组、Block 边界及不整除长度。
        test_x = cp.arange(test_n, dtype=cp.float32)  # 每轮创建连续输入。
        test_y = cp.ones(test_n, dtype=cp.float32)  # 第二输入使用相同长度与 dtype。
        test_out = cp.empty_like(test_x)  # 输出空间由 kernel 完整覆盖。
        if test_n > 0:  # 空输入无需启动 kernel，也不构造零 Block Grid。
            test_blocks = (test_n + 255) // 256  # 每个 Block 固定 256 个线程。
            vector_add((test_blocks,), (256,), (test_x, test_y, test_out, np.int32(test_n)))  # 显式匹配 int。
        expected_test = np.arange(test_n, dtype=np.float32) + 1  # 空数组的参照也为空。
        np.testing.assert_allclose(cp.asnumpy(test_out), expected_test)  # 逐轮验证输出。
    print("PASS: 6 种长度边界全部通过。")
else:
    print("SKIPPED: GPU 边界测试未执行。")

## 6. 同步计时：先定义哪些成本进入结果

第一项计时只覆盖 GPU 数组已经就绪后的矩阵乘法循环，不含输入上传和结果回传。循环仍包含输出分配及其内存池复用，不能称为单个底层 kernel 的独立耗时。


In [ ]:
if gpu_verified:  # 没有真实 GPU 时不生成计时数字。
    size = 1024  # 输入和输出均为 1024×1024。
    repeats = 10  # 统计连续 10 次矩阵乘法的平均设备时间。
    timed_x = cp.random.random((size, size), dtype=cp.float32)  # 数据直接生成在 GPU。
    for _ in range(3):  # 预热 cuBLAS、相关分配和执行路径。
        timed_result = timed_x @ timed_x.T
    cp.cuda.get_current_stream().synchronize()  # 在开始计时前等待预热完成。
    start_event = cp.cuda.Event()  # 创建 CUDA 起点事件。
    end_event = cp.cuda.Event()  # 创建 CUDA 终点事件。
    start_event.record()  # 把起点插入当前 CUDA Stream。
    for _ in range(repeats):  # 与预热保持相同运算和 dtype。
        timed_result = timed_x @ timed_x.T
    end_event.record()  # 把终点排在全部矩阵乘法之后。
    end_event.synchronize()  # 等待终点执行，避免只测到异步提交。
    mean_gpu_ms = cp.cuda.get_elapsed_time(start_event, end_event) / repeats  # 事件间毫秒除以次数。
    print(f"GPU 驻留数据的 matmul 平均时间: {mean_gpu_ms:.3f} ms")
else:
    print("SKIPPED: 没有可用 GPU，不能报告 GPU 性能。")

## 7. 端到端计时：把上传与回传也算进去

这一项包含主机到设备上传、GPU 运算和设备到主机回传；主机随机数组构造与 NumPy 正确性校验在计时范围之外。它与上一项回答的问题不同，不能直接取两者比值宣称 GPU 加速倍数。


In [ ]:
if gpu_verified:  # 与前一个计时单元使用相同设备。
    rng = np.random.default_rng(0)  # 固定 CPU 随机种子，便于复跑相同数据。
    host_x = rng.random((size, size), dtype=np.float32)  # 主机输入在计时前准备。
    cp.cuda.get_current_stream().synchronize()  # 排空前一单元的设备工作。
    wall_start = time.perf_counter()  # 从开始上传前记录墙钟时间。
    device_x = cp.asarray(host_x, blocking=True)  # 计入主机到设备的复制。
    device_result = device_x @ device_x.T  # 计入 GPU 矩阵乘法。
    host_result = cp.asnumpy(device_result)  # 默认阻塞，计入回传并等到结果可用。
    end_to_end_ms = (time.perf_counter() - wall_start) * 1000  # 得到本轮端到端毫秒。
    np.testing.assert_allclose(host_result, host_x @ host_x.T, rtol=1e-4, atol=1e-4)  # 校验不计入耗时。
    print(f"上传 + matmul + 回传，本轮端到端时间: {end_to_end_ms:.3f} ms")
else:
    print("SKIPPED: 端到端 GPU 路径未执行。")

## 后续实验

改变矩阵维度、传输次数、线程数时，每轮只改变一个变量，同时保留 dtype、数据是否已在显存、预热次数与同步方式。
没有进行 dtype、stride、显存预算或多 Stream 验证前，不把这个最小实验包装成通用 CUDA 算子库。
